# Train a TopK SAE on Qwen3.5-4B

**Goal:** ship the first public sparse autoencoder for the Qwen3.5 architecture.

**Target:** residual stream at layer 18 (mid-depth) of Qwen/Qwen3.5-4B.

**Config:** TopK SAE, k=64, d_sae=40960 (16× expansion), ~200M tokens from FineWeb-Edu.

**Hardware recommendation:**
- **T4 (Free tier)**: works but very slow — expect ~24h for 200M tokens. Use `--tokens 50_000_000` for a day-scale smoke test.
- **L4 / A100**: expected ~6-12h for the full 200M token run.
- **H100**: ~3-5h.

**Cost on a paid Colab runtime:** L4 ≈ $5-15, A100 ≈ $10-30, H100 ≈ $15-50 depending on runtime.

**Output:** a single `sae_final.pt` file plus metadata JSON, uploaded to HuggingFace Hub.

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print(f'torch {torch.__version__} | cuda {torch.version.cuda} | bf16 {torch.cuda.is_bf16_supported()}')

## 2. Install dependencies

`mechreward` is only needed for the final validation step. Training itself is pure torch + transformers.

In [ ]:
!pip install -q --upgrade pip
!pip install -q 'transformers>=4.44' accelerate datasets huggingface_hub safetensors tqdm
!pip install -q git+https://github.com/huggingface/transformers.git  # needed for Qwen3.5 support

## 3. HuggingFace login

Set your HF token (needed to download Qwen3.5-4B if gated, and to upload the trained SAE).

In [ ]:
import os
TOKEN = 'hf_YOUR_HF_TOKEN'  # <-- REPLACE WITH YOUR TOKEN
os.environ['HF_TOKEN'] = TOKEN
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
from huggingface_hub import login
login(token=TOKEN)
print('OK')

## 4. Fetch the training script

Pulls the latest version from the mechreward repo.

In [ ]:
!mkdir -p /content/mechreward && cd /content/mechreward && curl -sSL https://raw.githubusercontent.com/caiovicentino/mechreward/main/scripts/train_sae_qwen35.py -o train_sae_qwen35.py && head -5 train_sae_qwen35.py

## 5. Smoke test (10M tokens, ~30 min on L4)

Run a small version first to validate the full pipeline. Expect to see `var_exp` increase from ~0 to 0.7-0.9 over the run.

In [ ]:
!cd /content/mechreward && python3 train_sae_qwen35.py \
    --model Qwen/Qwen3.5-4B \
    --layer 18 \
    --d-sae 40960 \
    --k 64 \
    --tokens 10_000_000 \
    --batch-size 2048 \
    --micro-batch 4 \
    --max-length 256 \
    --warmup-steps 200 \
    --output-dir /content/sae_smoke

### Sanity check the smoke checkpoint

In [ ]:
import torch, json

ckpt = torch.load('/content/sae_smoke/sae_final.pt', map_location='cpu', weights_only=True)
print('Keys:', list(ckpt.keys()))
print(f"W_enc shape: {ckpt['W_enc'].shape}")
print(f"W_dec shape: {ckpt['W_dec'].shape}")
print(f"d_model: {ckpt['d_model']}")
print(f"d_sae:   {ckpt['d_sae']}")
print(f"k:       {ckpt['k']}")

with open('/content/sae_smoke/sae_final.json') as f:
    meta = json.load(f)
print('\nMetadata:')
print(json.dumps(meta, indent=2))

## 6. Full training run (200M tokens)

This is the real one. On an L4 expect ~6-12h; on A100 ~3-6h.

If the smoke test looked healthy (var_exp monotonically rising to > 0.7), kick this off. Otherwise debug first.

The `--hf-repo` flag uploads the final SAE to HuggingFace Hub automatically at the end.

In [ ]:
HF_REPO = 'caiovicentino/Qwen3.5-4B-SAE-L18-topk'  # <-- CHANGE IF NEEDED

!cd /content/mechreward && python3 train_sae_qwen35.py \
    --model Qwen/Qwen3.5-4B \
    --layer 18 \
    --d-sae 40960 \
    --k 64 \
    --tokens 200_000_000 \
    --batch-size 4096 \
    --micro-batch 8 \
    --max-length 512 \
    --warmup-steps 1000 \
    --output-dir /content/sae_final \
    --hf-repo $HF_REPO \
    --hf-token $TOKEN

## 7. Validate the trained SAE

Now use `mechreward` to test that the SAE actually works: encode some text, inspect the top features, and run the feature validator on candidate reasoning features.

In [ ]:
!pip install -q mechreward

In [ ]:
from mechreward.sae.topk_sae import load_topk_sae

sae = load_topk_sae(
    checkpoint='/content/sae_final/sae_final.pt',
    model_name='Qwen/Qwen3.5-4B',
    layer=18,
)
print(f'SAE loaded: d_model={sae.d_model}, d_sae={sae.d_sae}')
print(f'Device: {sae.device}')

In [ ]:
# Run a few probe texts through the model + SAE to find top-activating features
import torch
from transformers import AutoTokenizer, AutoModelForImageTextToText

tok = AutoTokenizer.from_pretrained('Qwen/Qwen3.5-4B', trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    'Qwen/Qwen3.5-4B',
    dtype=torch.bfloat16,
    device_map='cuda',
    trust_remote_code=True,
)
model.eval()

probes = {
    'step_reasoning': 'Step 1: First, I need to identify the variables.',
    'hedging': 'I think maybe the answer could possibly be around 5.',
    'confident': 'The answer is definitively 42, without any doubt.',
    'fact_recall': 'The capital of France is Paris.',
    'arithmetic': '17 times 23 equals 391.',
}

def get_activations(text: str) -> torch.Tensor:
    enc = tok(text, return_tensors='pt').to('cuda')
    with torch.inference_mode():
        out = model(**enc, output_hidden_states=True, return_dict=True)
    return out.hidden_states[19][0, -1]  # layer 18 output, last token

for name, text in probes.items():
    h = get_activations(text).float()
    acts = sae.encode(h.unsqueeze(0)).squeeze(0)
    top_vals, top_idx = acts.topk(10)
    top_ids = top_idx.tolist()
    print(f'\n{name}:')
    print(f'  {text}')
    print(f'  Top-10 feature IDs: {top_ids}')
    print(f'  Activations: {[f"{v:.2f}" for v in top_vals.tolist()]}')

## 8. Write the validated feature pack

Pick the features that cleanly discriminated between probes (e.g. the feature that fires strongly only on 'step_reasoning' but not on others), and save them to a JSON pack that mechreward can consume.

In [ ]:
from mechreward.features.catalog import Feature, FeaturePack, save_pack

# Replace with the IDs you identified above
validated_features = [
    Feature(feature_id=0, name='PLACEHOLDER', description='Set after analysis', weight=0.0),
]

pack = FeaturePack(
    name='qwen3.5-4b/reasoning_pack',
    version='0.1.1',
    model_name='Qwen/Qwen3.5-4B',
    release='mechreward/qwen3.5-4b-topk',
    sae_id='layer_18',
    description='First validated feature pack for Qwen3.5-4B.',
    features=validated_features,
)

save_pack(pack, '/content/qwen35_4b_reasoning_pack.json')
print('Pack saved. Upload to the mechreward repo via PR.')